# 01 | ทำความเข้าใจข้อมูลรีวิวโรงแรม

**คำถามตั้งต้น:** รีวิวในไฟล์บอกอะไรเกี่ยวกับประสบการณ์ของผู้เขียน และมีข้อจำกัดอะไรที่ต้องแก้หรือระบุไว้ก่อนสร้างกราฟ? หนึ่งแถวแทนหนึ่งรายการรีวิวในไฟล์

อ่าน [พจนานุกรมข้อมูล](../docs/data_dictionary.md) เพื่อดูทั้ง 13 ฟิลด์ โดยเฉพาะ **`rating` คือคะแนนที่ผู้รีวิวให้** ส่วน **`hotel_rating` คือระดับดาวของโรงแรมตามไฟล์**

In [ ]:
from pathlib import Path
import hashlib

import pandas as pd

# รองรับการเปิดจากโฟลเดอร์โปรเจกต์หรือจาก notebooks/
working_dir = Path.cwd()
root = working_dir.parent if working_dir.name == "notebooks" else working_dir
if not (root / "README.md").is_file() or not (root / "notebooks").is_dir():
    raise FileNotFoundError("Open the notebook from the portfolio folder or its notebooks/ folder.")

raw_name = "samplehotel - Sheet1.csv"
raw_path = root / "data" / "raw" / raw_name
if not raw_path.is_file():
    raise FileNotFoundError(f"Missing data/raw/{raw_name}. See README.md for setup.")
source_hash = hashlib.sha256(raw_path.read_bytes()).hexdigest()
print(f"Source: {raw_path.relative_to(root)}")
print(f"SHA-256: {source_hash}")

## ขนาด ตารางข้อมูล และค่าสูญหาย

ค่าว่างจริงนับจากช่องที่ว่างหรือมีแต่ช่องว่าง ส่วน `Not Specified` เป็นข้อความที่สื่อว่าไม่มีข้อมูลและนับแยก ตรวจแถวซ้ำแบบตรงทั้งแถวโดยใช้ข้อมูลที่อ่านจาก CSV

In [ ]:
raw = pd.read_csv(raw_path, keep_default_na=False, encoding="utf-8-sig")
as_text = raw.astype("string").apply(lambda col: col.str.strip())
blank = as_text.eq("")
placeholder = as_text.eq("Not Specified")
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]} | Exact repeated rows: {raw.duplicated().sum()}")
profile = pd.DataFrame({
    "inferred_dtype": raw.dtypes.astype(str),
    "blank": blank.sum(),
    "not_specified": placeholder.sum(),
    "unique_nonblank": raw.mask(blank).nunique(),
})
display(profile)

## คะแนนและค่าที่ไม่เปลี่ยนในชุดข้อมูล

ตรวจสถิติเชิงตัวเลขและการกระจายของคะแนนทุกระดับ แสดงจำนวนหมวดหมู่และตัวอย่างค่าที่พบบ่อยโดยไม่นำชื่อผู้รีวิวมาใช้เป็นกลุ่มวิเคราะห์

In [ ]:
number_cols = ["rating", "experience_year", "experience_month", "hotel_rating"]
numeric = raw[number_cols].apply(pd.to_numeric, errors="coerce")
display(numeric.describe().round(3))
display(numeric["rating"].value_counts().reindex(range(1, 6), fill_value=0).rename("reviews"))
for col in ["publish_platform", "travel_type", "clean_country", "name", "hotel_rating", "hotel_location"]:
    print(f"{col} ({raw[col].nunique()} unique)")
    display(raw[col].value_counts().head(6).rename("reviews"))
print("Invalid ratings:", (~numeric["rating"].isin(range(1, 6))).sum())

## วันที่เขียนรีวิวเทียบกับฟิลด์ Experience

ตรวจเทียบปีและเดือน **ทีละแถว** หลังแปลงวันที่ ทั้งหมดที่ตรงกันเป็นหลักฐานว่าฟิลด์เหล่านี้ซ้ำกับส่วนปี/เดือนของวันที่เขียนในไฟล์นี้ และยังไม่มีข้อมูลยืนยันวันเข้าพัก

In [ ]:
written = pd.to_datetime(raw["clean_written_date"], errors="coerce")
valid_date = written.notna()
display(pd.Series({
    "first_written_date": written.min(),
    "last_written_date": written.max(),
    "unparseable_dates": (~valid_date).sum(),
    "comparable_dates": valid_date.sum(),
    "year_mismatches": (written.dt.year.ne(numeric["experience_year"]) & valid_date).sum(),
    "month_mismatches": (written.dt.month.ne(numeric["experience_month"]) & valid_date).sum(),
}))
display(written.dt.to_period("M").value_counts().sort_index().rename("reviews_written"))

## คุณภาพข้อความ รีวิวซ้ำ และร่องรอยการถูกตัด

นับตัวอักษรจากข้อความดิบด้วย Python โดยไม่ตัดช่องว่างก่อนตรวจรูปแบบ 503 ตัวอักษรและลงท้ายด้วย `...` ร่องรอยนี้สอดคล้องกับการตัดข้อความ แต่ไม่มีฉบับเต็มให้ตรวจยืนยัน ตรวจข้อความซ้ำทั้งแบบตรงกันและแบบที่ลดช่องว่างและปรับตัวพิมพ์แล้ว

In [ ]:
text = raw["text"].astype("string")
text_normalized = text.str.strip().str.replace(r"\s+", " ", regex=True).str.casefold()
nonblank = text_normalized.ne("")
possible_truncation = text.str.len().eq(503) & text.str.endswith("...")
display(pd.Series({
    "blank_text": (~nonblank).sum(),
    "exact_repeated_nonblank_texts": text[nonblank].duplicated().sum(),
    "normalized_repeated_nonblank_texts": text_normalized[nonblank].duplicated().sum(),
    "max_characters": text.str.len().max(),
    "exactly_503_characters": text.str.len().eq(503).sum(),
    "503_characters_ending_in_three_dots": possible_truncation.sum(),
    "contains_thai_script": text.str.contains(r"[\u0E00-\u0E7F]").sum(),
}))
display(text.str.len().describe().round(2))

## เทียบกับข้อมูลอ้างอิงที่ได้รับ

คอลัมน์ `matches` แสดงว่าค่าที่คำนวณจากไฟล์ที่เปิดตรงกับตัวเลขอ้างอิงหรือไม่ ความตรงกันของตัวเลขสรุปใช้ตรวจความคาดหมายได้ แต่การยืนยันว่าเป็นไฟล์เดียวกันต้องเปรียบเทียบค่า SHA-256 ของทั้งสองไฟล์

In [ ]:
expected = {
    "rows": 1000, "columns": 13, "hotel": ("Amari Phuket",),
    "hotel_star_unique_values": 1, "hotel_location_unique_values": 1,
    "first_year": 2019, "last_year": 2022, "mean_rounded": 4.712, "five_star_reviews": 815,
    "blank_display_name": 3, "country_not_specified": 451, "travel_not_specified": 328,
    "duplicate_rows": 0, "possible_truncation": 361,
}
observed = {
    "rows": len(raw), "columns": raw.shape[1], "hotel": tuple(raw["name"].unique()),
    "hotel_star_unique_values": raw["hotel_rating"].nunique(),
    "hotel_location_unique_values": raw["hotel_location"].nunique(),
    "first_year": written.dt.year.min(), "last_year": written.dt.year.max(),
    "mean_rounded": round(numeric["rating"].mean(), 3), "five_star_reviews": numeric["rating"].eq(5).sum(),
    "blank_display_name": blank["displayName"].sum(),
    "country_not_specified": placeholder["clean_country"].sum(),
    "travel_not_specified": placeholder["travel_type"].sum(),
    "duplicate_rows": raw.duplicated().sum(), "possible_truncation": possible_truncation.sum(),
}
comparison = pd.DataFrame({"given": expected, "observed": observed})
comparison["matches"] = comparison["given"].eq(comparison["observed"])
display(comparison)

## ส่งต่อให้ขั้น Cleaning

- เก็บ `Unknown` เป็นหมวดหมู่แยก เพื่อรักษาจำนวนรีวิวและเห็นขนาดช่องว่างของข้อมูล
- แปลงวันที่และตรวจคะแนนก่อนวิเคราะห์ ใช้ปี/เดือนจากวันที่เขียนรีวิวเป็นแกนเวลา
- เก็บข้อความเดิมไว้และเพิ่มธงร่องรอยการถูกตัด เรื่องภาษาและความยาวข้อความจะถูกพิจารณาตอนวิเคราะห์ธีม
- ใช้ค่าคงที่สำหรับบอกบริบทของโรงแรมหนึ่งแห่ง